# Module 06: TurboQuant Stage 1 -- MSE-Optimal Quantization

**Time**: ~40 minutes

**Prerequisites**: Lloyd-Max (Module 02), FWHT (Module 03), RHT (Module 04), Beta distribution (Module 05)

**What you will learn**:
1. How the three ingredients combine into the full MSE quantizer (Algorithm 1)
2. A working toy implementation of the TurboQuant Stage 1 pipeline
3. How distortion scales with bit-width (the 4x-per-bit rule)
4. Why non-uniform quantization matters after rotation
5. How quantized inner products perform for nearest-neighbor search

---

You have built every piece individually. Now we assemble them.

In [ ]:
import sys; sys.path.insert(0, "..")
from src.helpers import (check_intuition, check_close, compare_distortions,
                         run_quantize_experiment, section_header)

import math
import numpy as np
import torch
from scipy.stats import norm
import plotly.graph_objects as go
from plotly.subplots import make_subplots

---
## 1. The Assembly: Three Ingredients, One Quantizer

We have three building blocks from earlier modules:

| Ingredient | Module | What it does |
|---|---|---|
| **Lloyd-Max** | 02 | Optimal scalar quantizer for a known distribution |
| **FWHT** | 03 | Fast orthogonal transform via butterfly |
| **RHT** | 04 | Random sign flips + FWHT = makes coordinates approximately Gaussian |

The key insight of TurboQuant Stage 1:

- **Lloyd-Max** gives you optimal scalar quantization *if you know the distribution*
- **RHT** makes every coordinate follow a known distribution (Beta, approximately Gaussian)
- **Therefore: RHT + Lloyd-Max = optimal vector quantizer!**

The norm is stored separately (a single float per vector), and the direction is quantized
coordinate-by-coordinate after rotation. That is the entire idea.

---
## 2. Algorithm 1: Step by Step

Here is the full pipeline, written as pseudocode:

```
QUANTIZE(x):
  1. Store norm:   gamma = ||x||
  2. Normalize:    x_unit = x / gamma
  3. Rotate:       x_rot = RHT(x_unit)
  4. For each coordinate: code_j = lloyd_max_quantize(x_rot[j])
  5. Return (codes, gamma)

DEQUANTIZE(codes, gamma):
  1. For each coordinate: x_rot_hat[j] = lloyd_max_centroid[code_j]
  2. Inverse rotate: x_unit_hat = RHT_inverse(x_rot_hat)
  3. Rescale:        x_hat = x_unit_hat * gamma
  4. Return x_hat
```

Five steps forward, four steps back. Let us trace through it with a concrete vector.

In [ ]:
section_header("Tracing Algorithm 1", "One vector through the full pipeline")

# --- Rebuild the pieces from earlier modules ---

def fwht(x):
    """Fast Walsh-Hadamard Transform (batched, operates on last dim)."""
    d = x.shape[-1]
    assert d > 0 and (d & (d - 1)) == 0, f"Length must be power of 2, got {d}"
    result = x.clone().float()
    h = 1
    while h < d:
        result = result.view(*result.shape[:-1], -1, 2 * h)
        left = result[..., :h]
        right = result[..., h:]
        result = torch.cat([left + right, left - right], dim=-1)
        result = result.view(*x.shape)
        h *= 2
    return result / math.sqrt(d)


def lloyd_max_normal(bits, max_iter=200, tol=1e-12):
    """Compute Lloyd-Max codebook for N(0,1)."""
    n_levels = 1 << bits
    centroids = np.linspace(-3, 3, n_levels)
    for _ in range(max_iter):
        boundaries = (centroids[:-1] + centroids[1:]) / 2.0
        new_centroids = np.zeros(n_levels)
        edges = np.concatenate([[-np.inf], boundaries, [np.inf]])
        for i in range(n_levels):
            lo, hi = edges[i], edges[i + 1]
            prob = norm.cdf(hi) - norm.cdf(lo)
            if prob < 1e-15:
                new_centroids[i] = (
                    (lo + hi) / 2
                    if np.isfinite(lo) and np.isfinite(hi)
                    else centroids[i]
                )
            else:
                new_centroids[i] = (norm.pdf(lo) - norm.pdf(hi)) / prob
        if np.max(np.abs(new_centroids - centroids)) < tol:
            centroids = new_centroids
            break
        centroids = new_centroids
    boundaries = (centroids[:-1] + centroids[1:]) / 2.0
    return boundaries, centroids


print("Pieces loaded: fwht, lloyd_max_normal")

In [ ]:
# Trace one vector through Algorithm 1, 2-bit quantization
torch.manual_seed(42)
d = 16
x = torch.randn(d)

print(f"Input vector x (d={d}):")
print(f"  x = {x.tolist()[:8]}...")
print()

# Step 1: Store the norm
gamma = x.norm()
print(f"Step 1 -- Store norm: gamma = ||x|| = {gamma:.4f}")

# Step 2: Normalize to unit vector
x_unit = x / gamma
print(f"Step 2 -- Normalize: ||x_unit|| = {x_unit.norm():.6f} (should be 1.0)")

# Step 3: Rotate with RHT (random signs + FWHT)
gen = torch.Generator().manual_seed(0)
signs = (torch.randint(0, 2, (d,), generator=gen) * 2 - 1).float()
x_signed = x_unit * signs
x_rot = fwht(x_signed)
print(f"Step 3 -- Rotate (RHT): ||x_rot|| = {x_rot.norm():.6f} (norm preserved)")
print(f"  x_rot = {x_rot.tolist()[:8]}...")

# Step 4: Lloyd-Max quantize each coordinate
bits = 2
boundaries, centroids = lloyd_max_normal(bits)
# Scale codebook by 1/sqrt(d) for the Gaussian approximation
scale = 1.0 / math.sqrt(d)
b_scaled = torch.from_numpy(boundaries * scale).float()
c_scaled = torch.from_numpy(centroids * scale).float()
codes = torch.searchsorted(b_scaled, x_rot)
print(f"Step 4 -- Quantize: codes = {codes.tolist()[:8]}...")
print(f"  (2 bits = {1 << bits} levels, codes range: [{codes.min()}, {codes.max()}])")
print()
print(f"Output: ({codes.shape[0]} codes, 1 norm) = {bits * d} bits + 32 bits for norm")

In [ ]:
# Now dequantize: reverse the pipeline
print("--- Dequantization ---")
print()

# Step 1: Centroid lookup
x_rot_hat = c_scaled[codes]
print(f"Step 1 -- Centroid lookup: x_rot_hat = {x_rot_hat.tolist()[:8]}...")

# Step 2: Inverse RHT (FWHT then sign flip)
x_unit_hat = fwht(x_rot_hat) * signs
print(f"Step 2 -- Inverse rotate: ||x_unit_hat|| = {x_unit_hat.norm():.6f}")

# Step 3: Rescale
x_hat = x_unit_hat * gamma
print(f"Step 3 -- Rescale: ||x_hat|| = {x_hat.norm():.4f} (original: {gamma:.4f})")
print()

# How good is the reconstruction?
mse = ((x - x_hat) ** 2).sum().item()
rel_err = (x - x_hat).norm().item() / x.norm().item()
print(f"Reconstruction MSE:      {mse:.6f}")
print(f"Relative error:          {rel_err:.4f} ({rel_err*100:.1f}%)")
print(f"Cosine similarity:       {torch.dot(x, x_hat).item() / (x.norm() * x_hat.norm()).item():.6f}")

---
## 3. Implement the Full Stage 1 Quantizer

Let us wrap the pipeline into a clean class.

In [ ]:
class ToyTurboQuantMSE:
    """Toy MSE-optimal vector quantizer: RHT + Lloyd-Max.
    
    This is the full Stage 1 pipeline from the TurboQuant paper.
    Uses our toy implementations of FWHT, RHT, and Lloyd-Max.
    """
    
    def __init__(self, dim, bits=2, seed=0):
        self.dim = dim
        self.bits = bits
        
        # Pad to next power of 2
        self.padded_dim = 1 << (dim - 1).bit_length() if dim > 0 else 1
        
        # Random signs for RHT
        gen = torch.Generator().manual_seed(seed)
        self.signs = (torch.randint(0, 2, (self.padded_dim,), generator=gen) * 2 - 1).float()
        
        # Lloyd-Max codebook scaled by 1/sqrt(padded_dim)
        boundaries, centroids = lloyd_max_normal(bits)
        scale = 1.0 / math.sqrt(self.padded_dim)
        self.boundaries = torch.from_numpy(boundaries * scale).float()
        self.centroids = torch.from_numpy(centroids * scale).float()
    
    def _rht_forward(self, x):
        """Apply RHT: pad, sign flip, FWHT."""
        if x.shape[-1] < self.padded_dim:
            padding = torch.zeros(*x.shape[:-1], self.padded_dim - x.shape[-1])
            x = torch.cat([x, padding], dim=-1)
        return fwht(x * self.signs)
    
    def _rht_inverse(self, x):
        """Apply inverse RHT: FWHT, sign flip, truncate."""
        x = fwht(x) * self.signs
        return x[..., :self.dim]
    
    def quantize(self, x):
        """Quantize vectors.
        
        Args:
            x: Input tensor of shape (n, dim).
        
        Returns:
            codes: Integer tensor of shape (n, padded_dim).
            norms: Float tensor of shape (n,).
        """
        # Step 1: Store norms
        norms = x.norm(dim=-1)
        
        # Step 2: Normalize (handle zero vectors)
        safe_norms = norms.clamp(min=1e-10)
        x_unit = x / safe_norms.unsqueeze(-1)
        
        # Step 3: Rotate
        x_rot = self._rht_forward(x_unit)
        
        # Step 4: Lloyd-Max quantize each coordinate
        codes = torch.searchsorted(self.boundaries, x_rot)
        
        return codes, norms
    
    def dequantize(self, codes, norms):
        """Reconstruct vectors from codes and norms.
        
        Args:
            codes: Integer tensor of shape (n, padded_dim).
            norms: Float tensor of shape (n,).
        
        Returns:
            Reconstructed tensor of shape (n, dim).
        """
        # Step 1: Centroid lookup
        x_rot_hat = self.centroids[codes]
        
        # Step 2: Inverse rotate
        x_unit_hat = self._rht_inverse(x_rot_hat)
        
        # Step 3: Rescale
        return x_unit_hat * norms.unsqueeze(-1)


# Quick sanity check
tq = ToyTurboQuantMSE(dim=16, bits=2, seed=0)
torch.manual_seed(42)
x_test = torch.randn(5, 16)
codes, norms = tq.quantize(x_test)
x_recon = tq.dequantize(codes, norms)

print(f"Input shape:  {x_test.shape}")
print(f"Codes shape:  {codes.shape}")
print(f"Norms shape:  {norms.shape}")
print(f"Recon shape:  {x_recon.shape}")
print(f"MSE: {((x_test - x_recon) ** 2).sum(dim=-1).mean():.6f}")

---
## 4. Test on Random Vectors

Generate 1000 random vectors in d=128 and measure MSE distortion at 1, 2, 3, and 4 bits.

In [ ]:
section_header("Distortion vs Bit-Width", "1000 vectors, d=128")

torch.manual_seed(123)
n_vectors = 1000
dim = 128
vectors = torch.randn(n_vectors, dim)

results = {}
for bits in [1, 2, 3, 4]:
    tq = ToyTurboQuantMSE(dim=dim, bits=bits, seed=0)
    
    def quantize_fn(v, _tq=tq):
        return _tq.quantize(v)
    
    def dequantize_fn(qout, _tq=tq):
        codes, norms = qout
        return _tq.dequantize(codes, norms)
    
    stats = run_quantize_experiment(
        quantize_fn, dequantize_fn, vectors, bits_label=f"({bits}-bit)"
    )
    results[f"{bits}-bit"] = stats["mse"]
    print()

---
## 5. Compare Against Paper Table 1

The TurboQuant paper (Table 1) reports MSE distortion for unit vectors.
Our vectors are not unit-norm, so we normalize the distortion by the
average squared norm to get a comparable number.

Your toy values should be in the right ballpark -- within 2-5x of the
paper values is fine. Differences come from using the Gaussian approximation
instead of the exact Beta codebook, and from finite sample effects.

In [ ]:
section_header("Comparison with Paper Table 1")

# Paper values for unit vector MSE distortion
paper_values = {
    "1-bit": 0.36,
    "2-bit": 0.117,
    "3-bit": 0.03,
    "4-bit": 0.009,
}

# Normalize our results: distortion on unit vectors
# Our MSE is on non-unit vectors. For unit vectors, divide by avg ||x||^2
avg_sq_norm = (vectors.norm(dim=-1) ** 2).mean().item()
normalized_results = {k: v / avg_sq_norm for k, v in results.items()}

print(f"Average ||x||^2 = {avg_sq_norm:.2f} (expected ~{dim} for d={dim})")
print()

compare_distortions(normalized_results, paper_values)

---
## 6. Visualize Distortion

Two things to see:
1. Distortion drops exponentially with bits (log scale)
2. The "4x per bit" rule: each additional bit cuts distortion by roughly 4x

In [ ]:
bits_list = [1, 2, 3, 4]
our_vals = [normalized_results[f"{b}-bit"] for b in bits_list]
paper_vals = [paper_values[f"{b}-bit"] for b in bits_list]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=[f"{b}-bit" for b in bits_list],
    y=our_vals,
    name="Our toy implementation",
    marker_color="steelblue",
    text=[f"{v:.4f}" for v in our_vals],
    textposition="outside",
))

fig.add_trace(go.Bar(
    x=[f"{b}-bit" for b in bits_list],
    y=paper_vals,
    name="Paper (Table 1)",
    marker_color="coral",
    text=[f"{v:.4f}" for v in paper_vals],
    textposition="outside",
))

fig.update_layout(
    title="MSE Distortion: Our Implementation vs Paper",
    xaxis_title="Bit-width",
    yaxis_title="MSE Distortion (unit vectors)",
    yaxis_type="log",
    template="plotly_white",
    height=500,
    barmode="group",
)

fig.show()

In [ ]:
# The 4x-per-bit rule
section_header("The 4x-per-bit Rule", "Each additional bit cuts distortion by ~4x")

print("Ratio of consecutive distortions (ours):")
for i in range(len(bits_list) - 1):
    b_lo = bits_list[i]
    b_hi = bits_list[i + 1]
    ratio = our_vals[i] / our_vals[i + 1]
    print(f"  {b_lo}-bit / {b_hi}-bit = {ratio:.2f}x")

print()
print("Ratio of consecutive distortions (paper):")
for i in range(len(bits_list) - 1):
    b_lo = bits_list[i]
    b_hi = bits_list[i + 1]
    ratio = paper_vals[i] / paper_vals[i + 1]
    print(f"  {b_lo}-bit / {b_hi}-bit = {ratio:.2f}x")

print()
print("The theoretical bound: D_mse <= (sqrt(3) * pi / 2) * (1 / 4^b)")
print("So distortion scales as 1/4^b -- each extra bit gives a 4x reduction.")

---
## 7. What the Quantized Data Looks Like

After rotation, values concentrate near zero. Non-uniform quantization
(Lloyd-Max) exploits this by putting more levels where the data is dense.

In [ ]:
section_header("Code Histogram", "2-bit quantization: 4 levels")

tq2 = ToyTurboQuantMSE(dim=dim, bits=2, seed=0)
codes_2bit, norms_2bit = tq2.quantize(vectors)

# Flatten all codes
all_codes = codes_2bit.flatten().numpy()

# Count occurrences of each level
n_levels = 1 << 2
counts = [(all_codes == i).sum() for i in range(n_levels)]
fractions = [c / len(all_codes) for c in counts]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=[f"Level {i}" for i in range(n_levels)],
    y=fractions,
    marker_color=["#e74c3c", "#3498db", "#3498db", "#e74c3c"],
    text=[f"{f:.1%}" for f in fractions],
    textposition="outside",
))

fig.update_layout(
    title="2-Bit Code Distribution: Middle Bins Dominate",
    xaxis_title="Quantization Level",
    yaxis_title="Fraction of Codes",
    template="plotly_white",
    height=400,
)
fig.show()

middle_fraction = fractions[1] + fractions[2]
print(f"Fraction in middle two bins: {middle_fraction:.1%}")
print(f"Fraction in outer two bins:  {1 - middle_fraction:.1%}")
print()
print("Most values land in the central bins because after rotation,")
print("coordinates follow an approximately Gaussian distribution")
print("concentrated near zero. This is WHY non-uniform quantization helps.")

In [ ]:
# Show the rotated coordinate distribution vs the codebook
section_header("Rotated Coordinates vs Codebook")

# Get the rotated values before quantization
norms_raw = vectors.norm(dim=-1)
safe_norms = norms_raw.clamp(min=1e-10)
v_unit = vectors / safe_norms.unsqueeze(-1)
v_rot = tq2._rht_forward(v_unit)

# Plot histogram of all rotated coordinates
rot_flat = v_rot.flatten().numpy()

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=rot_flat, nbinsx=100, name="Rotated coords",
    marker_color="steelblue", opacity=0.7,
    histnorm="probability density",
))

# Overlay the codebook boundaries and centroids
for b in tq2.boundaries.numpy():
    fig.add_vline(x=b, line_dash="dash", line_color="red", line_width=2)

for i, c in enumerate(tq2.centroids.numpy()):
    fig.add_vline(x=c, line_dash="dot", line_color="green", line_width=2,
                  annotation_text=f"c{i}" if i == 0 else None)

fig.update_layout(
    title="Rotated Coordinate Distribution + 2-Bit Lloyd-Max Codebook",
    xaxis_title="Value",
    yaxis_title="Density",
    template="plotly_white",
    height=400,
)
fig.add_annotation(
    x=0.95, y=0.95, xref="paper", yref="paper",
    text="Red dashed = boundaries<br>Green dotted = centroids",
    showarrow=False, bgcolor="white", bordercolor="gray", borderpad=4,
)
fig.show()

print("The distribution is concentrated near zero -- exactly what Lloyd-Max")
print("is designed for. The centroids cluster where the data is dense.")

---
## 8. Nearest-Neighbor Search Test

The whole point of quantizing vectors is to use them for similarity search.
Let us test: if we quantize 500 vectors and use quantized inner products
for nearest-neighbor search, how well does it work compared to exact search?

In [ ]:
section_header("Nearest-Neighbor Recall", "Quantized vs exact inner products")

torch.manual_seed(777)
n_db = 500
n_queries = 50
dim_nn = 128
k = 10  # recall@10

# Database and query vectors
db_vectors = torch.randn(n_db, dim_nn)
query_vectors = torch.randn(n_queries, dim_nn)

# Exact nearest neighbors (by inner product)
exact_scores = query_vectors @ db_vectors.T  # (n_queries, n_db)
_, exact_topk = exact_scores.topk(k, dim=1)  # (n_queries, k)

print(f"Database: {n_db} vectors, d={dim_nn}")
print(f"Queries:  {n_queries} vectors")
print(f"Metric:   recall@{k}")
print()

recall_results = {}
for bits in [1, 2, 3, 4]:
    tq = ToyTurboQuantMSE(dim=dim_nn, bits=bits, seed=42)
    
    # Quantize database
    db_codes, db_norms = tq.quantize(db_vectors)
    db_recon = tq.dequantize(db_codes, db_norms)
    
    # Quantize queries
    q_codes, q_norms = tq.quantize(query_vectors)
    q_recon = tq.dequantize(q_codes, q_norms)
    
    # Approximate nearest neighbors using reconstructed vectors
    approx_scores = q_recon @ db_recon.T
    _, approx_topk = approx_scores.topk(k, dim=1)
    
    # Compute recall@k: fraction of true top-k that appear in approximate top-k
    recall = 0.0
    for q in range(n_queries):
        true_set = set(exact_topk[q].tolist())
        approx_set = set(approx_topk[q].tolist())
        recall += len(true_set & approx_set) / k
    recall /= n_queries
    
    recall_results[bits] = recall
    print(f"  {bits}-bit: recall@{k} = {recall:.3f} ({recall*100:.1f}%)")

print()
print("Even at 2 bits per coordinate, recall is surprisingly good.")
print("At 3-4 bits, quantized search is nearly as good as exact search.")

In [ ]:
# Visualize recall vs bits
fig = go.Figure()

fig.add_trace(go.Bar(
    x=[f"{b}-bit" for b in recall_results],
    y=list(recall_results.values()),
    marker_color=["#e74c3c", "#f39c12", "#27ae60", "#2980b9"],
    text=[f"{v:.1%}" for v in recall_results.values()],
    textposition="outside",
))

fig.add_hline(y=1.0, line_dash="dash", line_color="gray",
              annotation_text="Perfect recall")

fig.update_layout(
    title=f"Nearest-Neighbor Recall@{k} vs Bit-Width",
    xaxis_title="Bit-width",
    yaxis_title=f"Recall@{k}",
    yaxis_range=[0, 1.1],
    template="plotly_white",
    height=450,
)
fig.show()

---
## 9. Connect to the Reference Code

Here is `TurboQuantMSE` from `reference-code/turboquant-torch/turboquant/mse_quantizer.py` --
the production version of what we just built.

In [ ]:
section_header("Reference Implementation", "turboquant/mse_quantizer.py")

print("""\
class TurboQuantMSE:
    def __init__(self, dim, bits=2, seed=0):
        self.dim = dim
        self.bits = bits
        self.rht = RandomizedHadamardTransform(dim, seed=seed)
        self.codebook = LloydMaxCodebook(bits, self.rht.padded_dim)

    def quantize(self, x):
        norms = torch.norm(x, dim=-1)
        safe_norms = norms.clamp(min=1e-10)
        x_unit = x / safe_norms.unsqueeze(-1)
        x_rot = self.rht.forward(x_unit)
        codes = self.codebook.quantize(x_rot)
        return MSEQuantizedOutput(codes=codes, norms=norms, bit_width=self.bits)

    def dequantize(self, output):
        x_rot = self.codebook.dequantize(output.codes)
        x_unit = self.rht.inverse(x_rot)
        return x_unit * output.norms.unsqueeze(-1)
""")

print("Three things to notice:")
print()
print("1. safe_norms = norms.clamp(min=1e-10)")
print("   Zero vectors would cause division by zero. Clamping the norm to a")
print("   tiny positive value avoids NaN and produces a near-zero reconstruction.")
print()
print("2. LloydMaxCodebook(bits, self.rht.padded_dim)")
print("   The codebook uses padded_dim, not the original dim. This matters")
print("   because the 1/sqrt(d) scaling in the codebook must match the actual")
print("   dimension after padding to a power of 2.")
print()
print("3. MSEQuantizedOutput is a NamedTuple")
print("   Bundles codes, norms, and bit_width into a single return value.")
print("   Clean API: dequantize() takes the output directly, no unpacking needed.")

In [ ]:
# Verify: our toy matches the reference
section_header("Verification", "Toy vs reference implementation")

sys.path.insert(0, "../reference-code/turboquant-torch")
from turboquant.mse_quantizer import TurboQuantMSE

torch.manual_seed(99)
test_vecs = torch.randn(100, 128)

for bits in [1, 2, 3, 4]:
    # Reference
    ref_tq = TurboQuantMSE(dim=128, bits=bits, seed=0)
    ref_out = ref_tq.quantize(test_vecs)
    ref_recon = ref_tq.dequantize(ref_out)
    ref_mse = ((test_vecs - ref_recon) ** 2).sum(dim=-1).mean().item()
    
    # Ours
    toy_tq = ToyTurboQuantMSE(dim=128, bits=bits, seed=0)
    toy_codes, toy_norms = toy_tq.quantize(test_vecs)
    toy_recon = toy_tq.dequantize(toy_codes, toy_norms)
    toy_mse = ((test_vecs - toy_recon) ** 2).sum(dim=-1).mean().item()
    
    ratio = toy_mse / ref_mse if ref_mse > 0 else float('nan')
    print(f"{bits}-bit: toy MSE = {toy_mse:.6f}, ref MSE = {ref_mse:.6f}, ratio = {ratio:.4f}")

print()
print("Ratios near 1.0 confirm our toy implementation matches the reference.")
print("Small differences come from codebook scaling (Gaussian vs exact Beta).")

---
## Exercise: Middle Bin Concentration

At 2 bits (4 levels), the two middle bins correspond to values closest to zero.
Since the rotated coordinates are approximately Gaussian and concentrated near zero,
the middle bins should get more than 50% of the codes.

**Task**: Quantize 1000 vectors at 2 bits and compute the fraction of codes
that map to the two middle bins (levels 1 and 2).

In [ ]:
section_header("Exercise", "Middle bin fraction at 2 bits")

torch.manual_seed(456)
exercise_vectors = torch.randn(1000, 128)

tq_ex = ToyTurboQuantMSE(dim=128, bits=2, seed=0)
ex_codes, ex_norms = tq_ex.quantize(exercise_vectors)

# Compute the fraction of codes in the two middle bins (levels 1 and 2)
all_codes_flat = ex_codes.flatten()
middle_mask = (all_codes_flat == 1) | (all_codes_flat == 2)
middle_fraction = middle_mask.float().mean().item()

print(f"Total codes: {all_codes_flat.shape[0]:,}")
print(f"Codes in middle bins (1 and 2): {middle_mask.sum().item():,}")
print(f"Fraction in middle bins: {middle_fraction:.4f} ({middle_fraction*100:.1f}%)")
print()

# For N(0, 1/d), the middle bins of a 2-bit Lloyd-Max quantizer should capture
# the probability mass between the two outer boundaries.
# For standard normal: P(-0.9816 < X < 0.9816) ~ 67.3%
# The boundaries scale by 1/sqrt(d), so this fraction stays the same.
check_intuition(
    middle_fraction,
    lambda x: x > 0.50,
    hint="After rotation, values are Gaussian near zero. The middle bins cover "
         "the high-density region, so they should capture well over 50%.",
    label="Middle bin fraction",
)

print()
print("This confirms the key insight: non-uniform quantization wins because")
print("most of the probability mass is near zero after rotation.")
print("A uniform quantizer would waste resolution on the sparse tails.")

---
## Summary

| Concept | Key Takeaway |
|---|---|
| Algorithm 1 | norm + normalize + RHT + Lloyd-Max = MSE-optimal vector quantizer |
| 4x-per-bit rule | Each additional bit cuts distortion by ~4x: D ~ 1/4^b |
| Code distribution | After rotation, coordinates are Gaussian -- middle bins dominate |
| Nearest-neighbor | Even 2-3 bits give surprisingly good recall for similarity search |
| Reference code | Same pipeline with safe_norms, padded_dim, NamedTuple output |

**What we built**: The complete Stage 1 (MSE-optimal) quantizer from the TurboQuant paper.
This is the foundation. Stage 2 adds **entropy coding** to get even better compression
by exploiting the non-uniform code distribution we observed in Section 7.